# EduRecom — Análisis Exploratorio y Evaluación del Sistema

Este notebook realiza el **análisis exploratorio de datos (EDA)** del dataset sintético de la plataforma educativa y evalúa el rendimiento del sistema de recomendación.

---
**Contenido:**
1. Carga de datos y descripción del dataset
2. Exploración de usuarios y cursos
3. Análisis de interacciones
4. Preparación de características para clustering
5. K-Means: método del codo e índice de Silhouette
6. Visualización de clusters (PCA 2D)
7. DBSCAN: detección de outliers
8. Filtrado colaborativo: Precision@K
9. Consideraciones éticas y privacidad

## 0. Configuración e importaciones

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, ROOT)

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', None)

DB_PATH = os.path.join(ROOT, 'database', 'edtech.db')
print('Base de datos:', DB_PATH)
print('Existe:', os.path.exists(DB_PATH))

## 1. Carga del dataset

In [ ]:
conn = sqlite3.connect(DB_PATH)
users        = pd.read_sql('SELECT * FROM users', conn)
courses      = pd.read_sql('SELECT * FROM courses', conn)
interactions = pd.read_sql('SELECT * FROM interactions', conn)
conn.close()

print(f'Usuarios:       {len(users):>6}')
print(f'Cursos:         {len(courses):>6}')
print(f'Interacciones:  {len(interactions):>6}')

## 2. Exploración de usuarios

In [ ]:
display(users.head())
print('\nDistribución de perfiles:')
display(users['profile'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de perfiles
users['profile'].value_counts().plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'))
axes[0].set_title('Distribución de perfiles de usuario')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Distribución de seniority
users['seniority'].value_counts().plot(kind='pie', ax=axes[1],
    autopct='%1.1f%%', colors=sns.color_palette('Set2'))
axes[1].set_title('Distribución de seniority')
axes[1].set_ylabel('')

# Distribución de edades
axes[2].hist(users['age'], bins=15, color=sns.color_palette('Set2')[2], edgecolor='white')
axes[2].set_title('Distribución de edades')
axes[2].set_xlabel('Edad')
axes[2].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## 3. Exploración de cursos e interacciones

In [ ]:
display(courses.describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

courses['category'].value_counts().plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'))
axes[0].set_title('Cursos por categoría')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

courses['level'].value_counts().plot(kind='pie', ax=axes[1],
    autopct='%1.1f%%', colors=sns.color_palette('Set2'))
axes[1].set_title('Distribución por nivel')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
display(interactions.describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de ratings
axes[0].hist(interactions['rating'], bins=20, color=sns.color_palette('Set2')[0], edgecolor='white')
axes[0].set_title('Distribución de ratings')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frecuencia')

# Distribución de progreso
axes[1].hist(interactions['progress_pct'], bins=20, color=sns.color_palette('Set2')[1], edgecolor='white')
axes[1].set_title('Distribución de progreso (%)')
axes[1].set_xlabel('Progreso (%)')

# Tasa de completados
comp_counts = interactions['completed'].value_counts().rename({0: 'No completado', 1: 'Completado'})
comp_counts.plot(kind='pie', ax=axes[2], autopct='%1.1f%%', colors=sns.color_palette('Set2'))
axes[2].set_title('Ratio de cursos completados')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Construcción de la matriz de características

In [ ]:
from models.clustering import build_user_feature_matrix, get_feature_columns

features = build_user_feature_matrix(interactions, courses)
feat_cols = get_feature_columns(features)

print(f'Dimensiones de la matriz: {features.shape}')
print(f'Características ({len(feat_cols)}):', feat_cols)
display(features.head())

In [ ]:
# Correlación entre características principales
main_feats = ['avg_rating', 'avg_progress', 'total_sessions', 'avg_session_duration', 'completed_ratio', 'n_courses']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(features[main_feats].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Correlación entre características principales')
plt.tight_layout()
plt.show()

## 5. K-Means: selección del número óptimo de clusters

Se usa el **método del codo** (inercia) y el **índice de Silhouette** para elegir `k`.

In [ ]:
X = features[feat_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k_range = range(2, 11)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

best_k = list(k_range)[int(np.argmax(silhouettes))]
best_sil = max(silhouettes)
print(f'Mejor k por Silhouette: k={best_k}  (score={best_sil:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2)
axes[0].set_title('Método del Codo — Inercia K-Means', fontweight='bold')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range), silhouettes, 'rs-', linewidth=2)
axes[1].axvline(x=best_k, color='green', linestyle='--', alpha=0.7, label=f'k óptimo={best_k}')
axes[1].set_title('Índice de Silhouette vs k', fontweight='bold')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. K-Means: ajuste y visualización (PCA 2D)

In [ ]:
N_CLUSTERS = 5   # Configuración del sistema
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
km_labels = km.fit_predict(X_scaled)
km_sil = silhouette_score(X_scaled, km_labels)

print(f'K-Means con k={N_CLUSTERS}')
print(f'  Silhouette score: {km_sil:.4f}')
print(f'  Inercia:          {km.inertia_:.2f}')

# Distribución de usuarios por cluster
unique, counts = np.unique(km_labels, return_counts=True)
for cl, cnt in zip(unique, counts):
    print(f'  Cluster {cl}: {cnt} usuarios')

In [ ]:
pca = PCA(n_components=2, random_state=42)
X2d = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame({'PC1': X2d[:, 0], 'PC2': X2d[:, 1], 'Cluster': km_labels.astype(str)})

fig, ax = plt.subplots(figsize=(9, 6))
palette = sns.color_palette('Set2', n_colors=N_CLUSTERS)
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='Cluster', palette=palette,
                alpha=0.75, s=60, ax=ax)
ax.set_title('Clusters K-Means — Proyección PCA 2D', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

print(f'Varianza explicada total: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 7. DBSCAN: detección de outliers

In [ ]:
from models.clustering import fit_dbscan

db_labels = fit_dbscan(X_scaled, eps=1.8, min_samples=5, pca_components=8)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = int((db_labels == -1).sum())
db_valid      = db_labels != -1

if db_valid.sum() > 1:
    db_sil = silhouette_score(X_scaled[db_valid], db_labels[db_valid])
    print(f'DBSCAN: clusters={n_clusters_db} | outliers={n_noise} | Silhouette={db_sil:.4f}')
else:
    print(f'DBSCAN: todos los puntos clasificados como ruido o 1 solo cluster')

# Visualización DBSCAN
pca2 = PCA(n_components=2, random_state=42)
X2d_db = pca2.fit_transform(X_scaled)

df_db = pd.DataFrame({'PC1': X2d_db[:, 0], 'PC2': X2d_db[:, 1],
                      'Cluster': [str(l) if l != -1 else 'Outlier' for l in db_labels]})

fig, ax = plt.subplots(figsize=(9, 6))
hue_order = sorted([c for c in df_db['Cluster'].unique() if c != 'Outlier'], key=int) + ['Outlier']
palette = {str(i): sns.color_palette('Set2')[i % 8] for i in range(n_clusters_db)}
palette['Outlier'] = 'gray'
sns.scatterplot(data=df_db, x='PC1', y='PC2', hue='Cluster', hue_order=hue_order,
                palette=palette, alpha=0.7, s=60, ax=ax)
ax.set_title('Clusters DBSCAN — Proyección PCA 2D', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8. Filtrado colaborativo — Precision@K

Se evalúa el sistema de recomendación usando **leave-one-out** sobre una muestra de usuarios.

In [ ]:
from models.collaborative import CollaborativeFilteringModel

model = CollaborativeFilteringModel(DB_PATH)
model.fit()
print(f'Matriz usuario×curso: {model.matrix.shape}')

# Evaluar Precision@K
metrics = model.evaluate(sample_size=100, k=5)
print(f'\nResultados Precision@5:')
print(f'  Media:             {metrics["precision_at_k"]:.4f} ({metrics["precision_at_k"]*100:.1f}%)')
print(f'  Desviación std:    ±{metrics["std"]:.4f}')
print(f'  Usuarios evaluados: {metrics["n_evaluated"]}')

In [ ]:
# Ejemplo de recomendaciones para un usuario concreto
sample_user = model.matrix.index[0]
recs = model.recommend(sample_user, top_n=5)

print(f'Top 5 recomendaciones para {sample_user}:')
display(recs[['course_id', 'name', 'category', 'level', 'predicted_rating', 'avg_rating']])

## 9. Resumen de métricas y comparativa de algoritmos

| Componente | Algoritmo | Métrica | Valor |
|---|---|---|---|
| Clustering | K-Means (k=5) | Silhouette | *(ver celda 6)* |
| Clustering | DBSCAN | Silhouette | *(ver celda 7)* |
| Recomendación | User-Based CF | Precision@5 | *(ver celda 8)* |

In [ ]:
# Tabla resumen
db_sil_val = silhouette_score(X_scaled[db_valid], db_labels[db_valid]) if db_valid.sum() > 1 else 0.0

summary_data = {
    'Componente': ['Clustering', 'Clustering', 'Recomendación'],
    'Algoritmo':  ['K-Means (k=5)', 'DBSCAN', 'User-Based CF'],
    'Métrica':    ['Silhouette', 'Silhouette', 'Precision@5'],
    'Valor':      [f'{km_sil:.4f}', f'{db_sil_val:.4f}', f'{metrics["precision_at_k"]*100:.1f}%'],
    'Interpretación': [
        'Buena cohesión si > 0.4',
        'Buena cohesión si > 0.4',
        'Porcentaje de recomendaciones relevantes'
    ]
}
display(pd.DataFrame(summary_data))

## 10. Consideraciones éticas y de privacidad

Este sistema implementa las siguientes medidas:

1. **Datos completamente sintéticos**: no se procesan datos personales reales de ningún usuario, eliminando riesgos de privacidad.
2. **Sin PII (Información de Identificación Personal)**: los identificadores de usuario son códigos anónimos (`U0001`, etc.).
3. **Transparencia en las recomendaciones**: la interfaz muestra el rating predicho junto al rating medio de la plataforma, permitiendo al usuario entender por qué se recomienda un curso.
4. **Cumplimiento RGPD**: en un entorno real, los datos deberían ser anonimizados o pseudoanonimizados antes de su procesamiento, con consentimiento explícito del usuario.
5. **Sesgo algorítmico**: el sistema de filtrado colaborativo puede perpetuar sesgos de popularidad. En producción, se recomienda combinar con filtrado basado en contenido (*hybrid filtering*) para mayor equidad.
6. **Derecho al olvido**: la arquitectura SQLite permite eliminar fácilmente los datos de un usuario específico sin afectar al resto del sistema.